## Workflow guide
The cells progress from loading Bronze tables to creating and validating each OMOP entity: person, visit occurrence, condition occurrence and observation period. Each validation runs before the relevant Delta table is saved.

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "clinical_portfolio"
BASE = f"{CATALOG}.{SCHEMA}"

patients = spark.table(f"{BASE}.bronze_patients")
encounters = spark.table(f"{BASE}.bronze_encounters")
conditions = spark.table(f"{BASE}.bronze_conditions")
condition_mapping = spark.table(f"{BASE}.bronze_condition_mapping")

display(encounters)

encounter_id,patient_id,encounter_start_date,encounter_end_date,encounter_type,_ingested_at,_source_file,_batch_id
E001,P001,2025-01-04,2025-01-04,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E002,P001,2025-03-15,2025-03-17,inpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E003,P002,2025-02-10,2025-02-10,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E004,P003,2025-02-28,2025-02-28,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E005,P004,2025-04-06,2025-04-08,inpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E006,P005,2025-04-19,2025-04-19,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E007,P006,2025-05-03,2025-05-03,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251
E008,P002,2025-05-22,2025-05-22,outpatient,2026-09-24T10:22:51.721Z,encounters.csv,20260924102251


### Load the Bronze inputs
The Silver transformation reads persisted Bronze tables rather than source files. This separation lets ingestion and standardization be rerun independently.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

gender_concepts = F.create_map(
    F.lit("M"), F.lit(8507),
    F.lit("F"), F.lit(8532),
)

person = (
    patients
    .select(
        F.abs(F.xxhash64("patient_id")).alias("person_id"),
        F.col("patient_id").alias("person_source_value"),
        F.col("birth_year").cast("int").alias("year_of_birth"),
        gender_concepts[F.col("gender_source_value")]
            .cast("int")
            .alias("gender_concept_id"),
        F.col("gender_source_value"),
    )
)

display(person)

person_id,person_source_value,year_of_birth,gender_concept_id,gender_source_value
7419654189246804363,P001,1978,8532,F
4905020018283288165,P002,1986,8507,M
7357811929381004785,P003,1994,8532,F
3215699940703695032,P004,1969,8507,M
3871179191872680772,P005,2001,8532,F
2763232168323479912,P006,1982,8507,M


In [0]:
invalid_persons = (
    person
    .filter(
        F.col("year_of_birth").isNull()
        | (F.col("year_of_birth") < 1900)
        | (F.col("year_of_birth") > 2026)
        | F.col("gender_concept_id").isNull()
    )
    .count()
)

assert invalid_persons == 0, f"Found {invalid_persons} invalid person records"
print("Person validation passed.")

Person validation passed.


In [0]:
(
    person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BASE}.person")
)

### Person entity
The next cells map source sex values to OMOP concepts and use a deterministic hash of the patient identifier. Hash-based IDs avoid a global sort, which is important for distributed Spark workloads.

In [0]:
visit_concepts = F.create_map(
    F.lit("outpatient"), F.lit(9202),
    F.lit("inpatient"), F.lit(9201)
)

person_lookup = person.select(
    "person_id",
    "person_source_value"
)

visit_occurrence = (
    encounters
    .join(
        person_lookup,
        encounters["patient_id"] == person_lookup["person_source_value"],
        "inner"
    )
    .select(
        F.abs(F.xxhash64("encounter_id")).alias("visit_occurrence_id"),
        F.col("person_id"),
        visit_concepts[F.col("encounter_type")]
        .cast("int")
        .alias("visit_concept_id"),
        F.to_date("encounter_start_date").alias("visit_start_date"),
        F.to_date("encounter_end_date").alias("visit_end_date"),
        F.col("encounter_id").alias("visit_source_value")
    )
)

display(visit_occurrence)

visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_end_date,visit_source_value
8377270851068047389,7419654189246804363,9202,2025-01-04,2025-01-04,E001
6128228633427423852,7419654189246804363,9201,2025-03-15,2025-03-17,E002
3743685860039125224,4905020018283288165,9202,2025-02-10,2025-02-10,E003
1175426780609763848,7357811929381004785,9202,2025-02-28,2025-02-28,E004
5268893089667615808,3215699940703695032,9201,2025-04-06,2025-04-08,E005
5188943524526667580,3871179191872680772,9202,2025-04-19,2025-04-19,E006
2287314808083437951,2763232168323479912,9202,2025-05-03,2025-05-03,E007
5943312001379122884,4905020018283288165,9202,2025-05-22,2025-05-22,E008


In [0]:
unmatched_visits = (
    encounters
    .join(
        person_lookup,
        encounters["patient_id"] == person_lookup["person_source_value"],
        "left_anti"
    )
    .count()
)

invalid_dates = (
    visit_occurrence
    .filter(F.col("visit_end_date") < F.col("visit_start_date"))
    .count()
)

assert unmatched_visits == 0, f"Found {unmatched_visits} visits without a person"
assert invalid_dates == 0, f"Found {invalid_dates} visits with invalid dates"

print("Visit validation passed.")

Visit validation passed.


In [0]:
(
    visit_occurrence.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BASE}.visit_occurrence")
)

display(spark.table(f"{BASE}.visit_occurrence"))

visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_end_date,visit_source_value
8377270851068047389,7419654189246804363,9202,2025-01-04,2025-01-04,E001
6128228633427423852,7419654189246804363,9201,2025-03-15,2025-03-17,E002
3743685860039125224,4905020018283288165,9202,2025-02-10,2025-02-10,E003
1175426780609763848,7357811929381004785,9202,2025-02-28,2025-02-28,E004
5268893089667615808,3215699940703695032,9201,2025-04-06,2025-04-08,E005
5188943524526667580,3871179191872680772,9202,2025-04-19,2025-04-19,E006
2287314808083437951,2763232168323479912,9202,2025-05-03,2025-05-03,E007
5943312001379122884,4905020018283288165,9202,2025-05-22,2025-05-22,E008


### Visit occurrence
Visits are created by joining encounters to the person lookup and mapping local encounter types to OMOP visit concepts. The preceding validation confirms that every visit has a person and a valid date interval.

In [0]:
display(
    condition_mapping.select(
        "source_condition_code",
        "standard_concept_id",
        "standard_concept_name",
        "mapping_status"
    )
)

source_condition_code,standard_concept_id,standard_concept_name,mapping_status
HTN,316866,Essential hypertension,standard
T2D,201826,Type 2 diabetes mellitus,standard
ASTHMA,317009,Asthma,standard
MIGRAINE,377821,Migraine,standard


In [0]:
standard_mappings = (
    condition_mapping
    .filter(F.col("mapping_status") == "standard")
)

unmapped_conditions = (
    conditions
    .join(
        standard_mappings,
        on="source_condition_code",
        how="left_anti"
    )
    .count()
)

assert unmapped_conditions == 0, (
    f"Found {unmapped_conditions} conditions without a standard OMOP mapping"
)

print("Condition mapping validation passed.")

Condition mapping validation passed.


In [0]:
encounter_lookup = encounters.select(
    "encounter_id",
    "patient_id"
)

visit_lookup = visit_occurrence.select(
    "visit_occurrence_id",
    "visit_source_value"
)

condition_enriched = (
    conditions
    .join(standard_mappings, on="source_condition_code", how="inner")
    .join(encounter_lookup, on="encounter_id", how="inner")
    .join(
        person_lookup,
        F.col("patient_id") == F.col("person_source_value"),
        "inner"
    )
    .join(
        visit_lookup,
        F.col("encounter_id") == F.col("visit_source_value"),
        "inner"
    )
)

display(condition_enriched)

encounter_id,source_condition_code,condition_event_id,condition_start_date,_ingested_at,_source_file,_batch_id,source_vocabulary,standard_concept_id,standard_concept_name,mapping_status,_ingested_at,_source_file,_batch_id,patient_id,person_id,person_source_value,visit_occurrence_id,visit_source_value
E001,HTN,C001,2025-01-04,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,316866,Essential hypertension,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P001,7419654189246804363,P001,8377270851068047389,E001
E002,T2D,C002,2025-03-15,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,201826,Type 2 diabetes mellitus,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P001,7419654189246804363,P001,6128228633427423852,E002
E002,HTN,C003,2025-03-15,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,316866,Essential hypertension,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P001,7419654189246804363,P001,6128228633427423852,E002
E003,ASTHMA,C004,2025-02-10,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,317009,Asthma,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P002,4905020018283288165,P002,3743685860039125224,E003
E004,MIGRAINE,C005,2025-02-28,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,377821,Migraine,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P003,7357811929381004785,P003,1175426780609763848,E004
E005,HTN,C006,2025-04-06,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,316866,Essential hypertension,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P004,3215699940703695032,P004,5268893089667615808,E005
E006,ASTHMA,C007,2025-04-19,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,317009,Asthma,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P005,3871179191872680772,P005,5188943524526667580,E006
E007,T2D,C008,2025-05-03,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,201826,Type 2 diabetes mellitus,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P006,2763232168323479912,P006,2287314808083437951,E007
E008,HTN,C009,2025-05-22,2026-09-24T10:22:56.192Z,conditions.csv,20260924102256,SYNTHETIC_LOCAL,316866,Essential hypertension,standard,2026-09-24T10:23:00.299Z,condition_mapping.csv,20260924102300,P002,4905020018283288165,P002,5943312001379122884,E008


In [0]:
condition_occurrence = (
    condition_enriched
    .select(
        F.abs(F.xxhash64("condition_event_id"))
        .alias("condition_occurrence_id"),
        F.col("person_id"),
        F.col("visit_occurrence_id"),
        F.col("standard_concept_id")
        .cast("int")
        .alias("condition_concept_id"),
        F.to_date("condition_start_date")
        .alias("condition_start_date"),
        F.col("source_condition_code")
        .alias("condition_source_value")
    )
)

display(condition_occurrence)

condition_occurrence_id,person_id,visit_occurrence_id,condition_concept_id,condition_start_date,condition_source_value
4583975396264843927,7419654189246804363,8377270851068047389,316866,2025-01-04,HTN
4817727097519339249,7419654189246804363,6128228633427423852,201826,2025-03-15,T2D
1721545497744665026,7419654189246804363,6128228633427423852,316866,2025-03-15,HTN
6384167949792045293,4905020018283288165,3743685860039125224,317009,2025-02-10,ASTHMA
3210767205019835406,7357811929381004785,1175426780609763848,377821,2025-02-28,MIGRAINE
639289959857423945,3215699940703695032,5268893089667615808,316866,2025-04-06,HTN
8234208537129745554,3871179191872680772,5188943524526667580,317009,2025-04-19,ASTHMA
2139690226847884731,2763232168323479912,2287314808083437951,201826,2025-05-03,T2D
4578636705409018139,4905020018283288165,5943312001379122884,316866,2025-05-22,HTN


In [0]:
invalid_conditions = (
    condition_occurrence
    .join(
        visit_occurrence.select(
            "visit_occurrence_id",
            "visit_start_date",
            "visit_end_date"
        ),
        on="visit_occurrence_id",
        how="inner"
    )
    .filter(
        ~F.col("condition_start_date").between(
            F.col("visit_start_date"),
            F.col("visit_end_date")
        )
    )
    .count()
)

assert invalid_conditions == 0, (
    f"Found {invalid_conditions} conditions outside their visit dates"
)

print("Condition date validation passed.")

Condition date validation passed.


In [0]:
condition_occurrence.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.condition_occurrence")

display(spark.table(f"{BASE}.condition_occurrence"))

condition_occurrence_id,person_id,visit_occurrence_id,condition_concept_id,condition_start_date,condition_source_value
4583975396264843927,7419654189246804363,8377270851068047389,316866,2025-01-04,HTN
4817727097519339249,7419654189246804363,6128228633427423852,201826,2025-03-15,T2D
1721545497744665026,7419654189246804363,6128228633427423852,316866,2025-03-15,HTN
6384167949792045293,4905020018283288165,3743685860039125224,317009,2025-02-10,ASTHMA
3210767205019835406,7357811929381004785,1175426780609763848,377821,2025-02-28,MIGRAINE
639289959857423945,3215699940703695032,5268893089667615808,316866,2025-04-06,HTN
8234208537129745554,3871179191872680772,5188943524526667580,317009,2025-04-19,ASTHMA
2139690226847884731,2763232168323479912,2287314808083437951,201826,2025-05-03,T2D
4578636705409018139,4905020018283288165,5943312001379122884,316866,2025-05-22,HTN


### Condition occurrence
Condition events are retained only when their source code has an approved standard mapping and they can be linked to both a patient and visit. This protects the analytic layer from unmapped or orphaned clinical events.

In [0]:
observation_period = (
    visit_occurrence
    .groupBy("person_id")
    .agg(
        F.min("visit_start_date").alias("observation_period_start_date"),
        F.max("visit_end_date").alias("observation_period_end_date")
    )
    .withColumn(
        "observation_period_id",
        F.abs(F.xxhash64("person_id"))
    )
    .select(
        "observation_period_id",
        "person_id",
        "observation_period_start_date",
        "observation_period_end_date"
    )
)

display(observation_period)

observation_period_id,person_id,observation_period_start_date,observation_period_end_date
3645087282003932193,3215699940703695032,2025-04-06,2025-04-08
6600667243263164327,2763232168323479912,2025-05-03,2025-05-03
2446329872212465907,3871179191872680772,2025-04-19,2025-04-19
5093836228448125674,7419654189246804363,2025-01-04,2025-03-17
1065753740330209936,7357811929381004785,2025-02-28,2025-02-28
5636882181170382917,4905020018283288165,2025-02-10,2025-05-22


In [0]:
invalid_periods = (
    observation_period
    .filter(
        F.col("observation_period_end_date")
        < F.col("observation_period_start_date")
    )
    .count()
)

assert invalid_periods == 0, f"Found {invalid_periods} invalid observation periods"
print("Observation period validation passed.")

Observation period validation passed.


In [0]:
observation_period.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.observation_period")

display(spark.table(f"{BASE}.observation_period"))

observation_period_id,person_id,observation_period_start_date,observation_period_end_date
3645087282003932193,3215699940703695032,2025-04-06,2025-04-08
6600667243263164327,2763232168323479912,2025-05-03,2025-05-03
2446329872212465907,3871179191872680772,2025-04-19,2025-04-19
5093836228448125674,7419654189246804363,2025-01-04,2025-03-17
1065753740330209936,7357811929381004785,2025-02-28,2025-02-28
5636882181170382917,4905020018283288165,2025-02-10,2025-05-22


### Observation period
An observation period spans a patient's first and last recorded visit. It is essential for cohort work because it defines the time during which the data can reasonably be considered observable.

# Silver OMOP standardization

This notebook transforms Bronze data into a focused OMOP CDM subset: person, visit occurrence, condition occurrence and observation period. It uses explicit mappings, stable hashed identifiers and checks for invalid demographics, unmatched visits and impossible dates before saving Delta tables.